<a href="https://colab.research.google.com/github/gabrieelsky/rps-cnn/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! git clone https://github.com/gabrieelsky/rps-cnn
! mv rps-cnn/* .
! rm -r rps-cnn/ sample_data/

! mkdir -p data/raw
! curl -L -o data/raw/rockpaperscissors.zip https://www.kaggle.com/api/v1/datasets/download/drgfreeman/rockpaperscissors
! unzip -q data/raw/rockpaperscissors.zip -d data/raw/
! rm data/raw/rockpaperscissors.zip
! mkdir saved_models

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from src.data_loader import create_dataloaders, get_class_mapping
from src.models import BaselineCNN, MicroResNet, MediumCNN
from src.train import train_model, run_grid_search
from src.evaluate import evaluate_model
from src.config import *

def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = get_device()
print(f"Hardware configuration: Using device '{device}'\n")

class_mapping = get_class_mapping(RAW_DATA_DIR)
num_classes = len(class_mapping)
print(f"{num_classes} classes found: {class_mapping}\n")

Hardware configuration: Using device 'mps'

3 classes found: {'paper': 0, 'rock': 1, 'scissors': 2}



In [ ]:
# # Define hyperparameter grid for Stratified K-Fold CV using run_grid_search
# param_grid = {
#     'lr': [1e-2, 1e-3, 3e-4],
#     'batch_size': [16, 32],
#     'dropout_rate': [0.3, 0.5],
#     'weight_decay': [0, 1e-4],
#     'epochs': [5] # To be set as 20 for the real tuning
# }

# print('Initiating Stratified K-Fold Grid Search (this may take a while)...')
# tuning_results = run_grid_search(BaselineCNN, param_grid, None, RAW_DATA_DIR, device=device, num_classes=num_classes, n_splits=3, seed=RANDOM_SEED, patience=3, min_delta=1e-4)
# tuning_results.to_csv('saved_models/grid_search_results.csv', index=False)
# print('Grid search complete. Top results:')
# print(tuning_results.head())

In [2]:
# Extract best configuration from tuning results
#best_config = tuning_results.iloc[0]
best_config = {
    'lr': 1e-3,
    'batch_size': 16,
    'dropout_rate': 0.5,
    'weight_decay': 1e-4,
    'epochs': 20
}

best_lr = float(best_config['lr'])
best_batch_size = int(best_config['batch_size'])
best_dropout = float(best_config.get('dropout_rate', 0.0))
best_weight_decay = float(best_config.get('weight_decay', 0.0))
best_epochs = int(best_config.get('epochs', 5))
final_epochs = 20  # Train longer for the final run
print(f"Selected best config: lr={best_lr}, batch_size={best_batch_size}, dropout={best_dropout}, weight_decay={best_weight_decay}")

Selected best config: lr=0.001, batch_size=16, dropout=0.5, weight_decay=0.0001


In [ ]:
train_loader, val_loader, test_loader, _ = create_dataloaders(RAW_DATA_DIR, batch_size=best_batch_size)

model = BaselineCNN(num_classes=num_classes, input_shape=(3, IMG_HEIGHT, IMG_WIDTH)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=best_lr, weight_decay=best_weight_decay)

print(f"\nStarting BaselineCNN training with the best parameters...")

model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=final_epochs,
    patience=5,
    min_delta=1e-4
)

print("\nEvaluating BaselineCNN on the test set...")
test_loss, all_preds, all_labels = evaluate_model(
    model=model,
    test_loader=test_loader,
    criterion=criterion,
    device=device,
    class_mapping=class_mapping,
)

save_path = os.path.join(MODELS_DIR, "baseline_cnn.pth")
torch.save(model.state_dict(), save_path)
print(f"\nBaselineCNN weights successfully saved to {save_path}")

In [ ]:
train_loader, val_loader, test_loader, _ = create_dataloaders(RAW_DATA_DIR, batch_size=best_batch_size)

model = MediumCNN(num_classes=num_classes, input_shape=(3, IMG_HEIGHT, IMG_WIDTH), dropout_rate=best_dropout).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=best_lr, weight_decay=best_weight_decay)

print(f"\nStarting MediumCNN training with the best parameters...")

model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=final_epochs,
    patience=5,
    min_delta=1e-4
)

print("\nEvaluating MediumCNN on the test set...")
test_loss, all_preds, all_labels = evaluate_model(
    model=model,
    test_loader=test_loader,
    criterion=criterion,
    device=device,
    class_mapping=class_mapping,
)

save_path = os.path.join(MODELS_DIR, "medium_cnn.pth")
torch.save(model.state_dict(), save_path)
print(f"\nMediumCNN weights successfully saved to {save_path}")

In [ ]:
train_loader, val_loader, test_loader, _ = create_dataloaders(RAW_DATA_DIR, batch_size=best_batch_size)

model = MicroResNet(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=best_lr, weight_decay=best_weight_decay)

print(f"\nStarting MicroResNet training with the best parameters...")

model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=final_epochs,
    patience=5,
    min_delta=1e-4
)

print("\nEvaluating MicroResNet on the test set...")
test_loss, all_preds, all_labels = evaluate_model(
    model=model,
    test_loader=test_loader,
    criterion=criterion,
    device=device,
    class_mapping=class_mapping,
)

save_path = os.path.join(MODELS_DIR, "micro_resnet.pth")
torch.save(model.state_dict(), save_path)
print(f"\nMicroResNet weights successfully saved to {save_path}")

In [ ]:
# only eval

# import importlib
# import src.evaluate as evaluate_module
# importlib.reload(evaluate_module)
# from src.evaluate import evaluate_model

# train_loader, val_loader, test_loader, _ = create_dataloaders(RAW_DATA_DIR, batch_size=best_batch_size)

# model = MicroResNet(num_classes=num_classes).to(device)
# state_dict = torch.load(os.path.join(MODELS_DIR, "convnet_best.pth"), map_location=device)
# model.load_state_dict(state_dict)
# criterion = nn.CrossEntropyLoss()

# test_loss, all_preds, all_labels = evaluate_model(
#     model=model,
#     test_loader=test_loader,
#     criterion=criterion,
#     device=device,
#     class_mapping=class_mapping,
# )

# print(f"\nEvaluation-only run completed. Test loss: {test_loss:.4f}")

In [ ]:
from src.models import MicroResNet
from src.webcam_utils import run_webcam_generalization_test

# Webcam-based generalization test.
# Show your hand in front of the camera, then press:
# - r for rock
# - p for paper
# - s for scissors
# Press q or Esc to stop early.

model = MicroResNet(num_classes=num_classes).to(device)
state_dict = torch.load(os.path.join(MODELS_DIR, "micro_resnet.pth"), map_location=device)
model.load_state_dict(state_dict)

# The function saves the captured images and a CSV summary under logs/webcam_generalization/.
webcam_results = run_webcam_generalization_test(
    model=model,
    device=device,
    class_mapping=class_mapping,
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH,
    samples_per_class=5,
    camera_index=0,
    save_dir=os.path.join(BASE_DIR, "logs", "webcam_generalization"),
)

webcam_results.head()


Webcam test started.
Press r/p/s to capture the current frame as rock/paper/scissors.
Press q or Esc to finish early.
No webcam samples were captured.


""
